## Part 5: Inference Function

`predict_podium(record)` predicts whether one driver will finish on the podium (top 3) in one race. It accepts the engineered **pre-race** features produced by Part 3 and uses the final trained model selected in Part 4. It does not rebuild features from raw driver/race IDs or retrain the model.

The function follows a strict sequence: **input validation → feature validation → model input preparation → prediction**. If the record is incomplete or invalid, it raises a clear error and stops before the model is called. This prevents a prediction from being produced from missing or unusable input.

In [ ]:
# LOCAL SETUP (dropped when the notebooks are merged)
import sys
sys.path.append("../src")
from f1_clean import POST_RACE_COLS
from IPython.utils.capture import capture_output
with capture_output():
    %run p4_models.ipynb

### 5.1 The function

The function uses `FEATURES` and `FEATURE_GROUPS` from Part 3, so the expected feature names and their training order stay synchronized with the modeling pipeline.

1. **Validate the record structure.** The input must be a dictionary for one driver-race record. Unexpected keys are rejected so IDs or unrelated columns cannot accidentally enter the model.
2. **Prevent target leakage.** Post-race fields and the target label are rejected because they are not known at the prediction cutoff: after qualifying and before the race starts.
3. **Check completeness.** A feature is missing if its key is absent or its value is `None` / `NaN`. If any required feature is missing, the function stops immediately.
4. **Validate supplied values.** Values must be finite numbers and satisfy basic domain checks for binary flags, positions, and non-negative counts/points.
5. **Prepare the model input.** Once validation succeeds, values are placed into a one-row DataFrame in exactly the same column order used for training. Preprocessing is not fitted again because it is already part of `FINAL_MODEL`.
6. **Predict.** The final pipeline returns the podium probability. `FINAL_THRESHOLD`, selected during model evaluation, converts it into the binary podium decision.

**Why reject incomplete records?** The fitted model pipeline includes imputation for its training/evaluation workflow, but an imputed value is not the observed pre-race value. This inference interface makes missing information explicit and does not silently substitute values or fallbacks.

In [ ]:
import numpy as np
import pandas as pd

# Map each final feature to its group so error messages explain where a missing
# input belongs (grid, qualifying, driver standings, or constructor standings).
GROUP_OF = {
    feature: group
    for group, features in FEATURE_GROUPS.items()
    for feature in features
}


def _is_missing(value):
    """Return True for None, NaN, or another scalar pandas missing-value marker."""
    try:
        result = pd.isna(value)
        # Feature values must be scalar; list/array values are handled as invalid.
        return bool(result) if np.isscalar(result) else False
    except (TypeError, ValueError):
        return False


def _validate_feature_value(feature, value):
    """
    Return a readable validation error, or None if the value is acceptable.
    These are basic domain checks based on the definitions in Part 3.
    """
    if isinstance(value, (bool, np.bool_)):
        return "must be numeric, not True/False"

    try:
        number = float(value)
    except (TypeError, ValueError, OverflowError):
        return "must be a finite number"

    if not np.isfinite(number):
        return "must be a finite number"

    # These features encode yes/no states, so only 0 and 1 are valid.
    binary_features = {
        "no_qual_time",
        "qg_q3",
        "qg_q2_only",
        "standings_missing",
        "cons_standings_missing",
    }
    if feature in binary_features and number not in (0.0, 1.0):
        return "must be 0 or 1"

    # Positions are one-based; zero and negative positions are not valid.
    position_features = {
        "grid_fixed",
        "qual_position",
        "pos_before",
        "cons_pos_before",
    }
    if feature in position_features:
        if number <= 0:
            return "must be greater than 0"
        if not number.is_integer():
            return "must be a whole number"

    # Points and a percentage gap cannot be negative.
    non_negative_features = {
        "pts_before",
        "cons_pts_before",
        "gap_to_pole_pct",
    }
    if feature in non_negative_features and number < 0:
        return "cannot be negative"

    return None


def predict_podium(record, model=None, threshold=None):
    """
    Predict the probability of a podium finish for one driver-race record.

    Pipeline:
        1. Validate input type and keys.
        2. Reject post-race information.
        3. Reject missing required features.
        4. Reject invalid feature values.
        5. Arrange a one-row model input in training feature order.
        6. Call the final model and apply its selected threshold.

    Missing or invalid input raises an explanatory error. The model is not
    called unless every required feature is present and valid.
    """
    # Stage 1: validate the input container before inspecting its contents.
    if not isinstance(record, dict):
        raise TypeError(
            "Invalid input: record must be a dictionary of engineered "
            f"pre-race features, not {type(record).__name__}."
        )

    non_string_keys = [key for key in record if not isinstance(key, str)]
    if non_string_keys:
        raise ValueError(
            "All input keys must be feature-name strings. Invalid key(s): "
            + ", ".join(repr(key) for key in non_string_keys)
        )

    # Use the final Part 4 model and threshold unless test doubles are supplied.
    selected_model = FINAL_MODEL if model is None else model
    selected_threshold = FINAL_THRESHOLD if threshold is None else threshold

    # Stage 2: reject post-race data and the target to prevent future information
    # from leaking into a prediction that should be made before race start.
    leaky = set(record) & (set(POST_RACE_COLS) | {"podium"})
    if leaky:
        raise ValueError(
            "Post-race information is not allowed. Remove these field(s): "
            + ", ".join(sorted(leaky))
        )

    # Only the model features are accepted. Driver/race IDs and descriptive
    # context are used upstream and are not part of the model input.
    unexpected = set(record) - set(FEATURES)
    if unexpected:
        raise ValueError(
            "Unexpected field(s): "
            + ", ".join(sorted(unexpected))
            + ". Supply only the engineered features listed in FEATURES."
        )

    # Stage 3: check for absent keys and explicit missing markers before any
    # preprocessing or model call. An incomplete record stops here.
    missing = [
        feature for feature in FEATURES
        if feature not in record or _is_missing(record[feature])
    ]
    if missing:
        grouped = [
            f"{feature} ({GROUP_OF.get(feature, 'unknown group')})"
            for feature in missing
        ]
        raise ValueError(
            f"Prediction stopped: {len(missing)} required feature(s) are missing: "
            + ", ".join(grouped)
            + f". Supply all {len(FEATURES)} features and try again. "
              "The model was not called."
        )

    # Stage 4: validate values only after confirming that every feature exists.
    invalid = {}
    values = {}
    for feature in FEATURES:
        problem = _validate_feature_value(feature, record[feature])
        if problem:
            invalid[feature] = problem
        else:
            values[feature] = float(record[feature])

    if invalid:
        details = [
            f"{feature}: {reason}"
            for feature, reason in invalid.items()
        ]
        raise ValueError(
            "Prediction stopped because feature value(s) are invalid: "
            + "; ".join(details)
            + ". Correct these values and try again. The model was not called."
        )

    # The decision threshold must be a valid probability cutoff.
    try:
        selected_threshold = float(selected_threshold)
    except (TypeError, ValueError, OverflowError) as error:
        raise ValueError("threshold must be a finite number between 0 and 1.") from error
    if not np.isfinite(selected_threshold) or not 0 <= selected_threshold <= 1:
        raise ValueError("threshold must be a finite number between 0 and 1.")

    # Stage 5: create the exact one-row, ordered input expected by the pipeline.
    # Preprocessing is already fitted inside FINAL_MODEL, so do not fit it here.
    X = pd.DataFrame(
        [[values[feature] for feature in FEATURES]],
        columns=FEATURES,
    )

    # Stage 6: run prediction only after all earlier validation stages pass.
    try:
        probabilities = selected_model.predict_proba(X)
        probability = float(probabilities[0, 1])
    except Exception as error:
        raise RuntimeError(
            "The final model failed during prediction. Check that it is fitted "
            "and uses the same FEATURES list supplied by Part 3."
        ) from error

    if not np.isfinite(probability) or not 0 <= probability <= 1:
        raise RuntimeError(
            f"The model returned an invalid podium probability: {probability!r}."
        )

    return {
        "status": "ok",
        "podium_probability": probability,
        "podium_prediction": int(probability >= selected_threshold),
        "missing_features": [],
        "invalid_features": [],
        "message": "Prediction completed successfully; all required features were present and valid.",
    }

### 5.2 Example A: Complete Record

This example selects a validation record for which all final model features are present. The actual finishing position is printed only to identify the historical example; it is not included in the input dictionary.

A successful call returns the estimated podium probability and the binary decision produced using the threshold selected in Part 4.

In [ ]:
def describe_row(row):
    driver = tables["drivers"].set_index("driverId").loc[row["driverId"], "surname"]
    race = tables["races"].set_index("raceId").loc[row["raceId"], "name"]
    finish = outcomes.set_index(["raceId", "driverId"]).loc[(row["raceId"], row["driverId"]), "positionOrder"]
    return f"{driver}, {race} {row['year']} (actual finish: P{int(finish)})"

complete_rows = table[(table["split"] == "validation") & table[FEATURES].notna().all(axis=1)]
row_A = complete_rows.iloc[0]
record_A = row_A[FEATURES].to_dict()

print(describe_row(row_A))
print("input:", record_A)
print("output:", predict_podium(record_A))

All required features are supplied, so this record follows the successful path through validation, model-input preparation, and prediction. The probability is an estimate from the final model, not a guarantee of the actual result. Overall model performance is evaluated separately in Part 4 using validation and unseen test data.

### 5.3 Example B: Incomplete Record

This example selects a validation record with at least one missing final feature. Missing fields are omitted from the input dictionary to represent an incomplete feature record.

The function should identify the missing feature names and their groups, raise `ValueError`, and stop before calling the model. This demonstrates how the inference interface handles incomplete input rather than silently imputing values.

In [ ]:
# Select a validation record that has at least one missing final feature.
incomplete_rows = table[
    (table["split"] == "validation")
    & ~table[FEATURES].notna().all(axis=1)
]
assert not incomplete_rows.empty, (
    "No incomplete validation row is available to demonstrate input error handling."
)

row_B = incomplete_rows.iloc[0]
record_B = {
    feature: value
    for feature, value in row_B[FEATURES].to_dict().items()
    if pd.notna(value)
}

print(describe_row(row_B))
print("input:", record_B)

try:
    predict_podium(record_B)
except ValueError as error:
    print("Prediction correctly refused.")
    print("Reason:", error)
    expected_missing = [
        feature for feature in FEATURES if pd.isna(row_B[feature])
    ]
    assert all(feature in str(error) for feature in expected_missing)
    assert "model was not called" in str(error)
else:
    raise AssertionError("The incomplete record should have raised ValueError.")

### 5.4 Validation Checks

The checks below cover both the normal inference path and expected failure cases:

* Complete validation rows must produce the same probability as a direct call to the final model with the same ordered features.
* Incomplete rows must raise `ValueError` and identify every missing feature.
* A tracking model verifies that missing input is rejected before `predict_proba` is called.
* Invalid numeric values, post-race columns, unexpected fields, and non-dictionary inputs must be rejected with clear errors.

In [ ]:
val = table[table["split"] == "validation"]
is_complete = val[FEATURES].notna().all(axis=1)

# Complete rows: the wrapper must agree with a direct call to the final pipeline.
val_ok = val[is_complete]
assert not val_ok.empty, "No complete validation rows are available for this check."

direct = FINAL_MODEL.predict_proba(val_ok[FEATURES])[:, 1]
results_ok = [
    predict_podium(row[FEATURES].to_dict())
    for _, row in val_ok.iterrows()
]
one_by_one = np.array([result["podium_probability"] for result in results_ok])

max_difference = float(np.max(np.abs(direct - one_by_one)))
print(
    "complete validation rows:", len(val_ok),
    "| largest probability difference from direct model call:", max_difference
)
assert all(result["status"] == "ok" for result in results_ok)
assert np.allclose(direct, one_by_one)

# Incomplete rows: verify each is rejected and every missing field is named.
val_bad = val[~is_complete]
for _, row in val_bad.iterrows():
    record = {
        feature: value
        for feature, value in row[FEATURES].to_dict().items()
        if pd.notna(value)
    }
    try:
        predict_podium(record)
    except ValueError as error:
        expected_missing = [
            feature for feature in FEATURES if pd.isna(row[feature])
        ]
        assert all(feature in str(error) for feature in expected_missing)
        assert "model was not called" in str(error)
    else:
        raise AssertionError(
            "An incomplete validation record unexpectedly produced a prediction."
        )

print(
    "incomplete validation rows:", len(val_bad),
    "| all were rejected before prediction"
)

# A spy model proves that missing input is rejected before predict_proba is called.
class ModelMustNotBeCalled:
    def predict_proba(self, X):
        raise AssertionError("predict_proba must not run for an incomplete record.")

incomplete_record = {
    feature: value
    for feature, value in row_B[FEATURES].to_dict().items()
    if pd.notna(value)
}
try:
    predict_podium(incomplete_record, model=ModelMustNotBeCalled())
except ValueError:
    print("early-abort check: passed; model was not called")
else:
    raise AssertionError("Incomplete input was not rejected.")

# Invalid feature values must be refused before prediction.
try:
    predict_podium({**record_A, "grid_fixed": "front"})
except ValueError as error:
    assert "grid_fixed" in str(error)
    print("invalid feature value correctly refused:", error)
else:
    raise AssertionError("A non-numeric grid_fixed value should have been rejected.")

# Post-race fields are forbidden because they would leak information from the outcome.
for col in ["positionOrder", "points", "status", "podium"]:
    try:
        predict_podium({**record_A, col: 1})
    except ValueError as error:
        assert col in str(error)
    else:
        raise AssertionError(f"Post-race field {col} should have been rejected.")
print("post-race leakage checks: passed")

# Unrecognized fields and wrong input types must also fail clearly.
try:
    predict_podium({**record_A, "unexpected_column": 1})
except ValueError as error:
    assert "unexpected_column" in str(error)
    print("unexpected field correctly refused:", error)
else:
    raise AssertionError("Unexpected input fields should have been rejected.")

try:
    predict_podium(None)
except TypeError as error:
    print("wrong input type correctly refused:", error)
else:
    raise AssertionError("A non-dictionary input should have been rejected.")

### 5.5 Missing-Feature Analysis

This section reports how many validation records have at least one missing final feature and which features are missing most often. It helps show how often the strict inference input contract would require upstream data to be completed before prediction.

The analysis is descriptive: it does not change the feature set or model, and it does not use the test set to make input-policy decisions.

In [ ]:
share = len(val_bad) / len(val)
print(f"validation rows with missing features: {len(val_bad)} of {len(val)} ({share:.1%}); complete rows: {len(val_ok)}")

missing_counts = val[FEATURES].isna().sum()
missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)
display(
    missing_counts.rename("rows with this feature missing")
    .to_frame()
    .assign(group=lambda d: d.index.map(GROUP_OF))
)


### 5.6 Limitations

* The function accepts engineered features from Part 3; it does not construct them from raw race tables or driver/race identifiers.
* A prediction requires all final model features to be present and valid. Missing values are reported and rejected rather than silently imputed.
* Basic validation can detect missing, non-numeric, non-finite, and clearly out-of-domain values, but it cannot verify that upstream feature engineering is factually correct.
* The returned probability is the final model's estimate of a podium finish, not a guarantee of the actual race outcome.